In [ ]:
import struct
import numpy as np
import math
from enum import Enum
import custom_math
import random

#Parsing the binary format
with open("/Users/abhinavarora/Desktop/Machine Learning/Neural Network/MNIST handwritten /train-images-idx3-ubyte/train-images-idx3-ubyte", "rb") as f:
    magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
    images = np.frombuffer(f.read(), dtype=np.uint8)
    images = images.reshape(num, rows * cols)

#Loading in the labels
with open("/Users/abhinavarora/Desktop/Machine Learning/Neural Network/MNIST handwritten /train-labels-idx1-ubyte/train-labels-idx1-ubyte", "rb") as f:
    magic, num = struct.unpack(">II", f.read(8))
    labels = np.frombuffer(f.read(), dtype=np.uint8)

In [ ]:
#Dimension of the vector
img_dimension = rows * cols
#Each image flattened into a vector and put into a matrix
image_input_matrix = [[0] * len(images) for _ in range(img_dimension)]

for column in range(len(images)):
    for row in range(len(images[0])):
        #Normalizing to fit every value between 0 and 1 to solve vanishing gradients
        image_input_matrix[row][column] = images[row][column]/255

In [ ]:
class ActivationFunctions:
    #Required to know the number of neurons since activation functions will be vector operations 
    def __init__(self, input_matrix):
        self.input_matrix = input_matrix

    def sigmoid(self, input_val):
        exponent = math.exp(-input_val)
        return (1/(1+exponent))
    
    def ReLU(self, input_val):
        if input_val > 0:
            return input_val
        return 0

    
class FunctionType(Enum):
    RELU = 1
    SIGMOID = 2

#This is a layer of neurons with specific number of neurons that is a hyper parameter
#As well as the number of inputs this layer will take
class Layer:
    def __init__(self, neuron_num, input_size, function_type: FunctionType):
        self.function_type = function_type
        self.neuron_num = neuron_num
        self.input_size = input_size
        #Generates a random number in the normal distribution with mean 0 and std 0.01. This results in values between
        #-0.03 and 0.03. This is required since we want to break symmetry in each layer of a neural network and not 
        #make them identical
        self.parameters = [[random.Random(float).gauss(0, 0.01)] * input_size for _ in range(neuron_num)]
        self.bias = [[0] * 1 for _ in range(neuron_num)]
    
    def execute_function(self, input_val):
        #Initialising the class of the different activation functions
        functions = ActivationFunctions()
        if self.function_type == FunctionType.RELU:
            return functions.ReLU(input_val)
        if self.function_type == FunctionType.SIGMOID:
            return functions.sigmoid(input_val)
    
    #Returns the omega * x + b
    #Bias automatically gets broadcasted in this function
    def linearize(self, parameters, input, bias):
        parameter_input_product = custom_math.matrix_with_matrix_multiplication(parameters, input)
        #Broadcast
        #This is the number of rows of the broadcasted bias matrix
        dimension = len(parameter_input_product[0])
        broadcasted_matrix = [[0] * len(dimension) for _ in range(parameter_input_product)]
        for row in range(len(broadcasted_matrix)):
            for col in range(len(broadcasted_matrix[0])):
                broadcasted_matrix[row][col] = bias[row]
        return custom_math.matrix_addition_and_sub(parameter_input_product, broadcasted_matrix, "add")

    #The softmax function is only for the last layer's output, by default it will be set to 0. However, the underlying
    #hypothesis function will not change
    def hypothesis(self, linear, softmax=False):
        linear_copy = [[0] * len(linear[0]) for _ in range(len(linear))]
        for row in range(len(linear)):
            for col in range(len(linear[0])):
                linear_copy[row][col] = self.execute_function(linear[row][col])
        
        if not softmax:
            return linear_copy
        else:
            for col in range(len(linear[0])):
                #Calculating the total exponential sum for each ouptut
                exponential_sum = 0
                for row in range(len(linear)):
                    exponential_sum += math.exp(linear_copy[row][col])
                
                #Applying the softmax formula
                for row in range(len(linear)):
                    linear_copy[row][col] = math.exp(linear_copy[row][col])/exponential_sum
        
        return linear_copy
        

In [ ]:
class Network:
    def __init__(self, layer_num, neurons_in_layers, initial_input):
        #This initialises the number of layers
        self.number_of_layers = layer_num
        #This is a list that specifies the number of neurons in each layer 
        self.neurons_in_layers = neurons_in_layers
        self.inital_input = initial_input
        #This is the array that stores the actual layer objects
        self.layers = []
        #Initialising the layers
        for i in range(len(self.number_of_layers)):
            #This is specifically for the first layer. This is because the input_size is the dimension of the vector of each training example
            if i == 0:
                layer = Layer(self.neurons_in_layers[i], len(self.inital_input), FunctionType.SIGMOID)
            #For the other layers, the input size the number of neurons of the previous layer since each neuron outputs a single number
            else:
                layer = Layer(self.neurons_in_layers[i], self.neurons_in_layers[i-1], FunctionType.SIGMOID)
            
            self.layers.append(layer)


    
